In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/saas_customers_raw.csv", parse_dates=["Subscription_Date", "Renewal_Date", "Cancellation_Date"])
print(df.shape)
df.info()

(35040, 22)
<class 'pandas.DataFrame'>
RangeIndex: 35040 entries, 0 to 35039
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Customer_ID               35040 non-null  str           
 1   Age                       35040 non-null  int64         
 2   Country                   35040 non-null  str           
 3   Industry                  34690 non-null  str           
 4   Company_Size              35040 non-null  str           
 5   Subscription_Plan         35040 non-null  str           
 6   Subscription_Date         35040 non-null  datetime64[us]
 7   Renewal_Date              25565 non-null  datetime64[us]
 8   Monthly_Fee               35040 non-null  float64       
 9   Discount_Percent          35040 non-null  float64       
 10  Payment_Method            34515 non-null  str           
 11  Marketing_Channel         35040 non-null  str           
 12  Support_Tickets  

In [2]:
print("Duplicate rows before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate rows after:", df.duplicated().sum())
print("New shape:", df.shape)

Duplicate rows before: 40
Duplicate rows after: 0
New shape: (35000, 22)


In [3]:
print("Rows with negative usage hours:", (df["Usage_Hours_Monthly"] < 0).sum())

median_usage = df.loc[df["Usage_Hours_Monthly"] >= 0, "Usage_Hours_Monthly"].median()
print("Median usage hours (valid rows only):", median_usage)

df.loc[df["Usage_Hours_Monthly"] < 0, "Usage_Hours_Monthly"] = median_usage

print("Rows with negative usage hours after fix:", (df["Usage_Hours_Monthly"] < 0).sum())

Rows with negative usage hours: 15
Median usage hours (valid rows only): 13.8
Rows with negative usage hours after fix: 0


In [4]:
print("Missing before:")
print(df[["Industry", "Payment_Method"]].isna().sum())

df["Industry"] = df["Industry"].fillna("Unknown")
df["Payment_Method"] = df["Payment_Method"].fillna(df["Payment_Method"].mode()[0])

print("\nMissing after:")
print(df[["Industry", "Payment_Method"]].isna().sum())

Missing before:
Industry          350
Payment_Method    525
dtype: int64

Missing after:
Industry          0
Payment_Method    0
dtype: int64


In [5]:
print("Missing before:", df["Customer_Satisfaction"].isna().sum())

median_satisfaction = df["Customer_Satisfaction"].median()
print("Median satisfaction:", median_satisfaction)

df["Customer_Satisfaction"] = df["Customer_Satisfaction"].fillna(median_satisfaction)

print("Missing after:", df["Customer_Satisfaction"].isna().sum())

Missing before: 700
Median satisfaction: 6.8
Missing after: 0


In [6]:
# Renewal_Date should be missing ONLY for Churned customers
mismatch_1 = df[(df["Customer_Status"] == "Active") & (df["Renewal_Date"].isna())]
print("Active customers missing Renewal_Date (should be 0):", len(mismatch_1))

# Cancellation_Date should be missing ONLY for Active customers
mismatch_2 = df[(df["Customer_Status"] == "Churned") & (df["Cancellation_Date"].isna())]
print("Churned customers missing Cancellation_Date (should be 0):", len(mismatch_2))

Active customers missing Renewal_Date (should be 0): 0
Churned customers missing Cancellation_Date (should be 0): 0


In [7]:
print("Final shape:", df.shape)
print("\nMissing values remaining:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("\nNegative usage hours:", (df["Usage_Hours_Monthly"] < 0).sum())
print("\nAge range:", df["Age"].min(), "-", df["Age"].max())
print("Monthly_Fee range:", df["Monthly_Fee"].min(), "-", df["Monthly_Fee"].max())

Final shape: (35000, 22)

Missing values remaining:
 Renewal_Date          9465
Cancellation_Date    25535
dtype: int64

Duplicate rows: 0

Negative usage hours: 0

Age range: 18 - 65
Monthly_Fee range: 13.5 - 286.33


In [8]:
df.to_csv("../data/processed/saas_customers_cleaned.csv", index=False)
print("Saved cleaned dataset:", df.shape)

Saved cleaned dataset: (35000, 22)
